In [1]:
# utils.py

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay
from PIL import Image
from ultralytics import YOLO

In [1]:
biomarker_questions = {
    'age': 'Enter your Age',
    'blood_pressure': 'Enter your blood pressure eg.80',
    'specific_gravity': 'Enter specific gravity eg.1.5',
    'albumin': 'Enter your albumin level eg.1',
    'sugar': 'Enter your sugar level in urine eg.2',
    'red_blood_cells': 'Do you have red blood cells in urine? Yes/No',
    'pus_cell': 'Do you have pus cells ? Yes/No',
    'pus_cell_clumps': 'Do you have pus cell clumps ? Yes/No',
    'bacteria': 'Do you have bacteria in your urine? Yes/No',
    'blood_glucose_random': 'Enter your blood sugar level ',
    'blood_urea': 'Enter your blood urea level ',
    'serum_creatinine': 'Enter your serum creatinine level in  mg/dL?',
    'sodium': 'Enter your sodium level mEq/L',
    'potassium': 'Enter your potassium level mEq/L',
    'haemoglobin': 'Enter your hemoglobin level mEq/L',
    'packed_cell_volume': 'Enter your packed cell volume %',
    'white_blood_cell_count': 'Enter your white blood cell count in cells/μL',
    'red_blood_cell_count': 'Enter your red blood cell count in million cells/μL',
    'hypertension': 'Do you have hypertension? Yes/No',
    'diabetes_mellitus': 'Do you have diabetes mellitus? Yes/No',
    'coronary_artery_disease': 'Do you have coronary artery disease? Yes/No',
    'appetite': 'Do you have poor appetite? Yes/No',
    'peda_edema': 'Do you have pedal edema? Yes/No',
    'anemia': 'Do you have anemia? Yes/No'
}
column_mappings = {
    'red_blood_cells': {'normal': 0, 'abnormal': 1},
    'pus_cell': {'normal': 0, 'abnormal': 1},
    'pus_cell_clumps': {'notpresent': 0, 'present': 1},
    'bacteria': {'notpresent': 0, 'present': 1},
    'hypertension': {'no': 0, 'yes': 1},
    'diabetes_mellitus': {'no': 0, 'yes': 1},
    'coronary_artery_disease': {'no': 0, 'yes': 1},
    'appetite': {'good': 0, 'poor': 1},
    'peda_edema': {'no': 0, 'yes': 1},
    'anemia': {'no': 0, 'yes': 1},
    'class':{'ckd':0,'not ckd':1}
}

def load_data_train(file_path):
    Data=file_path
    df=pd.read_csv(Data)
    df.drop('id', axis = 1, inplace = True)
    df.columns = ['age', 'blood_pressure', 'specific_gravity', 'albumin', 'sugar', 'red_blood_cells', 'pus_cell',
              'pus_cell_clumps', 'bacteria', 'blood_glucose_random', 'blood_urea', 'serum_creatinine', 'sodium',
              'potassium', 'haemoglobin', 'packed_cell_volume', 'white_blood_cell_count', 'red_blood_cell_count',
              'hypertension', 'diabetes_mellitus', 'coronary_artery_disease', 'appetite', 'peda_edema',
              'anemia', 'class']
    df['diabetes_mellitus'].replace(to_replace = {'\tno':'no','\tyes':'yes',' yes':'yes'},inplace=True)
    df['coronary_artery_disease'] = df['coronary_artery_disease'].replace(to_replace = '\tno', value='no')
    df['class'] = df['class'].replace(to_replace = {'ckd\t': 'ckd', 'notckd': 'not ckd'})
    # converting necessary columns to numerical type
    df['packed_cell_volume'] = pd.to_numeric(df['packed_cell_volume'], errors='coerce')
    df['white_blood_cell_count'] = pd.to_numeric(df['white_blood_cell_count'], errors='coerce')
    df['red_blood_cell_count'] = pd.to_numeric(df['red_blood_cell_count'], errors='coerce')
    
    # Apply custom mappings to the selected columns
    df.replace(column_mappings, inplace=True)
    return df


def load_data_predict(file_path):
    Data=file_path
    df=pd.read_csv(Data)
    if 'id' in df.columns:
        df.drop('id', axis=1, inplace=True)

    df.columns = ['age', 'blood_pressure', 'specific_gravity', 'albumin', 'sugar', 'red_blood_cells', 'pus_cell',
              'pus_cell_clumps', 'bacteria', 'blood_glucose_random', 'blood_urea', 'serum_creatinine', 'sodium',
              'potassium', 'haemoglobin', 'packed_cell_volume', 'white_blood_cell_count', 'red_blood_cell_count',
              'hypertension', 'diabetes_mellitus', 'coronary_artery_disease', 'appetite', 'peda_edema',
              'anemia']
    df['diabetes_mellitus'].replace(to_replace = {'\tno':'no','\tyes':'yes',' yes':'yes'},inplace=True)
    df['coronary_artery_disease'] = df['coronary_artery_disease'].replace(to_replace = '\tno', value='no')
    # converting necessary columns to numerical type
    df['packed_cell_volume'] = pd.to_numeric(df['packed_cell_volume'], errors='coerce')
    df['white_blood_cell_count'] = pd.to_numeric(df['white_blood_cell_count'], errors='coerce')
    df['red_blood_cell_count'] = pd.to_numeric(df['red_blood_cell_count'], errors='coerce')
    # Apply custom mappings to the selected columns
    df.replace(column_mappings, inplace=True)
    return df


def preprocess_data(data,for_prediction=False):
    
    # Convert Series to numpy array
    if not for_prediction:
        X = data.drop("class", axis=1)
        y = data["class"]
    else:
        X = data  # Assuming data contains only features for prediction
    
    
    # Define default values for each column
    column_defaults = {
        'age': 51.483376,
        'blood_pressure': 76.469072,
        'specific_gravity': 1.017408,
        'albumin': 0,  # Example albumin level
        'sugar': 0,  # Example sugar level
        'red_blood_cells': 0,  # Example red blood cell presence (1 for normal, 0 for abnormal)
        'pus_cell': 1,  # Example pus cell presence (1 for normal, 0 for abnormal)
        'pus_cell_clumps': 0,  # Example pus cell clumps presence (1 for present, 0 for absent)
        'bacteria': 0,  # Example bacteria presence (1 for present, 0 for absent)
        'blood_glucose_random': 120,
        'blood_urea': 57.425722,
        'serum_creatinine': 0.450142,
        'sodium': 137.528754,
        'potassium': 4.627244,
        'haemoglobin': 12.526437,
        'packed_cell_volume': 45,  # Example packed cell volume percentage
        'white_blood_cell_count': 7000,  # Example white blood cell count cells/μL
        'red_blood_cell_count': 5,  # Example red blood cell count million cells/μL
        'hypertension': 0,  # Example hypertension presence (1 for yes, 0 for no)
        'diabetes_mellitus': 0,  # Example diabetes mellitus presence (1 for yes, 0 for no)
        'coronary_artery_disease': 0,  # Example coronary artery disease presence (1 for yes, 0 for no)
        'appetite': 0,  # Example appetite (0 for good, 1 for poor)
        'peda_edema': 0,  # Example pedal edema presence (1 for yes, 0 for no)
        'anemia': 0  # Example anemia presence (1 for yes, 0 for no)
    }

    # Replace null values with custom default values using an if-else condition
    for col_name in column_defaults.keys():
        default_value = column_defaults[col_name]
        X[col_name] = X[col_name].fillna(default_value)
        
    df=data 

    # Replace null values with default values using fillna
    df.fillna(column_defaults, inplace=True)

    # Define the maximum column width for alignment
    max_col_width = max(len(biomarker) for biomarker in column_defaults.keys())

    # Define a threshold for numeric values that indicate "normal"
    normal_thresholds = {
        'blood_pressure': 80.0,
        'specific_gravity': 1.025,
        'blood_glucose_random': 140.0,
        'blood_urea': 50.0,
        'serum_creatinine': 1.2,
        'sodium': 135.0,
        'potassium': 3.5,
        'haemoglobin': 13.5,
        'packed_cell_volume': 45.0,  # normal packed cell volume percentage
        'white_blood_cell_count': 11000.0,  # normal white blood cell count cells/μL
        'red_blood_cell_count': 5.0,  # normal red blood cell count million cells/μL
    }

    # Loop through each row in the DataFrame
    for index, row in df.iterrows():
        # print(f"Biomarker Values for Row {index + 1}:")
        for biomarker, default_value in column_defaults.items():
            value = row[biomarker]
            if biomarker in ['red_blood_cells', 'pus_cell', 'pus_cell_clumps', 'bacteria']:
                # Biomarkers with 1 for normal and 0 for abnormal
                result = "normal" if value == 1 else "abnormal"
            elif biomarker in ['hypertension', 'diabetes_mellitus', 'coronary_artery_disease', 'appetite', 'peda_edema', 'anemia']:
                # Biomarkers with 1 for yes and 0 for no
                result = "yes" if value == 1 else "no"
            elif biomarker in normal_thresholds:
                # Check if the numeric value is within the "normal" range
                normal_threshold = normal_thresholds[biomarker]
                result = "normal" if value <= normal_threshold else "abnormal"
                # Append the numerical value to the result
                value_str=str(value).rjust(6)
                result = f"{value_str} ({result})"
            else:
                result = str(value)  # Numeric biomarkers
            # Format the output with aligned ":"
            print(f"{biomarker.ljust(max_col_width)}: {result.rjust(6)}")

    if not for_prediction:
        return X, y
    else:
        return X
    
def predict_labels(df=None,ImgTest=None):
    
    #Model Load
    ImgModel=YOLO('../best.pt')
    BioMarker_path = 'trained_model2.pt' 
    Biomodel = KidneyCNN()
    Biomodel.load_state_dict(torch.load(BioMarker_path))

    if df is None and ImgTest is None:
        print("Please give the Image and BioMarker data")
        return 0
   
    if df is not None:
        # Convert DataFrame to PyTorch tensor
        X_new_tensor = torch.tensor(df.to_numpy(), dtype=torch.float32)

        # Perform predictions
        with torch.no_grad():
            outputs = Biomodel(X_new_tensor)
            _, predicted = torch.max(outputs, 1)

        # Map the predicted class indices to class labels
        class_mapping = {0: 'Chronic Kidney Disease', 1: 'Not Chronic Kidney Disease'}
        prediction1 = [class_mapping[idx] for idx in predicted.tolist()]
        print(prediction1[0])

    if ImgTest is not None:
        result=ImgModel(ImgTest, imgsz=640, conf=0.5)
        names = ImgModel.names

        for r in result:
            for c in r.boxes.cls:
                prediction2 =names[int(c)]
        for r in result:
            im_array = r.plot()  # plot a BGR numpy array of predictions
            im = Image.fromarray(im_array[..., ::-1])  # RGB PIL image
            im.show()  # show image
            im.save('results.jpg')  # save image

        print(prediction2)
    return 0


class KidneyCNN(nn.Module):
    def __init__(self, input_size=24, num_classes=2):
        super(KidneyCNN, self).__init__()
        self.conv1 = nn.Conv1d(1, 32, kernel_size=3)
        self.conv2 = nn.Conv1d(32, 64, kernel_size=3)
        self.fc1 = nn.Linear(64 * (input_size - 4), 128)
        self.fc2 = nn.Linear(128, num_classes)

    def forward(self, x):
        x = x.unsqueeze(1)
        x = torch.relu(self.conv1(x))
        x = torch.relu(self.conv2(x))
        x = x.view(x.size(0), -1)
        x = torch.relu(self.fc1(x))
        x = self.fc2(x)
        return x


NameError: name 'nn' is not defined